In [0]:
dbutils.widgets.combobox("catalog", "workspace", ["workspace", "dbr_dev"])
catalog_name = dbutils.widgets.get("catalog")

dbutils.widgets.text("silver_table", "mens_volleyball_clean")

silver_table = dbutils.widgets.get("silver_table")
silver_table_name = f"{catalog_name}.trezio2005_silver.{silver_table}"

In [0]:
df = spark.table(silver_table_name)

Data quailty rules

In [0]:
#CONSTRAINTS
#scores can't be negative
spark.sql(f"ALTER TABLE {silver_table_name} DROP CONSTRAINT IF EXISTS check_positive_score")
spark.sql(f"""
          ALTER TABLE {silver_table_name} 
          ADD CONSTRAINT check_positive_score CHECK (T1_Score >= 0 AND T2_Score >= 0)
          """)

spark.sql(f"ALTER TABLE {silver_table_name} DROP CONSTRAINT IF EXISTS check_positive_sum")
spark.sql(f"""
          ALTER TABLE {silver_table_name} 
          ADD CONSTRAINT check_positive_sum CHECK (T1_Sum >= 0 AND T2_Sum >= 0)
          """)

#statistics can't be negative
spark.sql(f"ALTER TABLE {silver_table_name} DROP CONSTRAINT IF EXISTS check_positive_srv_stat")
spark.sql(f"""
          ALTER TABLE {silver_table_name} 
          ADD CONSTRAINT check_positive_srv_stat CHECK (T1_Srv_Sum >= 0 AND T2_Srv_Sum >= 0 AND T1_Srv_Err >= 0 AND T2_Srv_Err >= 0 AND T1_Srv_Ace >= 0 AND T2_Srv_Ace >= 0)
          """)

spark.sql(f"ALTER TABLE {silver_table_name} DROP CONSTRAINT IF EXISTS check_positive_rec_stat")
spark.sql(f"""
          ALTER TABLE {silver_table_name}
          ADD CONSTRAINT check_positive_rec_stat CHECK (T1_Rec_Sum >= 0 AND T2_Rec_Sum >= 0 AND T1_Rec_Err >= 0 AND T2_Rec_Err >= 0 AND T1_Rec_Pos >= 0 AND T2_Rec_Pos >= 0 AND T1_Rec_Perf >= 0 AND T2_Rec_Perf >= 0)
          """)

spark.sql(f"ALTER TABLE {silver_table_name} DROP CONSTRAINT IF EXISTS check_positive_att_stat")
spark.sql(f"""
          ALTER TABLE {silver_table_name}
          ADD CONSTRAINT check_positive_att_stat CHECK (T1_Att_Sum >= 0 AND T2_Att_Sum >= 0 AND T1_Att_Err >= 0 AND T2_Att_Err >= 0 AND T1_Att_Blk >= 0 AND T2_Att_Blk >= 0 AND T1_Att_Kill >= 0 AND T2_Att_Kill >= 0 AND T1_Att_Kill_Perc >= 0 AND T2_Att_Kill_Perc >= 0 AND T1_Att_Eff >= 0 AND T2_Att_Eff >= 0)
          """)

#winner has to be known
spark.sql(f"ALTER TABLE {silver_table_name} DROP CONSTRAINT IF EXISTS check_for_winner")
spark.sql(f"""
          ALTER TABLE {silver_table_name}
          ADD CONSTRAINT check_for_winner CHECK (Winner = 0 OR Winner = 1) 
          """)



Classic partitioning - dividies the data into multiple folders. Quick and reliable when used corectly, but when partitioned by a key that has a lot of unique values 
can result in creation of a lot of small files that ultimatelly leads to slower read access

ZOrdering - uses Z-order curve algortihm to partition data into groups that are logically similar to each other. When data is queried thanks to z ordering only relevant data is looked and which results in quick query. The downside is that the z-order curve is computational heavy and has to be run everytime a new data is beeing added

Liquid Clustering - works incrementally, clusters only the newly added data and thanks to that is quick and cost effective. The best option at the time



In [0]:
#Liquid clustring by most used keys
spark.sql(f"ALTER TABLE {silver_table_name} CLUSTER BY (Date, Team_1, Team_2)")
spark.sql(f"OPTIMIZE {silver_table_name}")


#Cleaning the old files using the vacuum function
spark.sql(f"VACUUM {silver_table_name} RETAIN 168 HOURS")